# Habitat Headless Test

Quick verification that habitat-sim + habitat-lab work in the uv env.

**Kernel:** uv-managed Python (`uv run jupyter lab`)

---
## 1. Import Checks

In [ ]:
import habitat_sim
print(f"habitat-sim: {habitat_sim.__version__}")

import habitat
print(f"habitat-lab: {habitat.__version__}")

import numpy as np
import matplotlib.pyplot as plt
print("All imports OK")

---
## 2. Test Scene Load + RGB-D Rendering

In [ ]:
import habitat_sim
import numpy as np
import matplotlib.pyplot as plt

cfg = habitat_sim.SimulatorConfiguration()
cfg.scene_id = "data/scene_datasets/habitat-test-scenes/skokloster-castle.glb"

# RGB sensor
rgb_spec = habitat_sim.CameraSensorSpec()
rgb_spec.uuid = "color_sensor"
rgb_spec.sensor_type = habitat_sim.SensorType.COLOR
rgb_spec.resolution = [256, 256]

# Depth sensor
depth_spec = habitat_sim.CameraSensorSpec()
depth_spec.uuid = "depth_sensor"
depth_spec.sensor_type = habitat_sim.SensorType.DEPTH
depth_spec.resolution = [256, 256]

agent_cfg = habitat_sim.agent.AgentConfiguration()
agent_cfg.sensor_specifications = [rgb_spec, depth_spec]

sim = habitat_sim.Simulator(habitat_sim.Configuration(cfg, [agent_cfg]))
obs = sim.get_sensor_observations()

print(f"color_sensor: {obs['color_sensor'].shape} {obs['color_sensor'].dtype}")
print(f"depth_sensor: {obs['depth_sensor'].shape} {obs['depth_sensor'].dtype}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(obs["color_sensor"][:, :, :3])
axes[0].set_title("RGB")
axes[0].axis("off")
axes[1].imshow(obs["depth_sensor"], cmap="plasma")
axes[1].set_title("Depth")
axes[1].axis("off")
plt.tight_layout()
plt.show()

---
## 3. Basic Navigation Loop

In [ ]:
agent = sim.get_agent(0)
action_names = list(agent.agent_config.action_space.keys())
print(f"Available actions: {action_names}")

# Execute a sequence of actions
action_seq = ["move_forward", "turn_left", "move_forward", "move_forward", "turn_right", "move_forward"]
frames = [obs]

for action in action_seq:
    obs = sim.step(action)
    frames.append(obs)

# Visualize first 4 frames
n_show = min(4, len(frames))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 6))
labels = ["reset"] + action_seq
for i in range(n_show):
    axes[0, i].imshow(frames[i]["color_sensor"][:, :, :3])
    axes[0, i].set_title(labels[i])
    axes[0, i].axis("off")
    axes[1, i].imshow(frames[i]["depth_sensor"], cmap="plasma")
    axes[1, i].set_title(f"depth {i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

state = agent.get_state()
print(f"Final position: {state.position}")
print(f"Final rotation: {state.rotation}")

sim.close()
print("Sim closed.")

---
## 4. ObjectNav Episode Load (val_mini)

Requires HM3D scene files (Matterport token). If scenes unavailable, loads episode metadata only.

In [ ]:
import json, gzip
from pathlib import Path

DATA_DIR = Path("data/datasets/objectnav/hm3d/objectnav_hm3d_v2")
SCENE_DIR = Path("data/scene_datasets/hm3d")

# Load val_mini episodes
content_dir = DATA_DIR / "val_mini" / "content"
if content_dir.exists():
    content_files = sorted(content_dir.glob("*.json.gz"))
    episodes = []
    for cf in content_files:
        with gzip.open(cf, "rt") as f:
            data = json.load(f)
        episodes.extend(data.get("episodes", []))
    print(f"Loaded {len(episodes)} val_mini episodes from {len(content_files)} scene files")
    
    ep = episodes[0]
    print(f"\nFirst episode:")
    print(f"  scene_id: {ep['scene_id']}")
    print(f"  target: {ep['object_category']}")
    print(f"  geodesic_distance: {ep['info']['geodesic_distance']:.2f} m")
    
    # Check if scene files exist
    has_scenes = SCENE_DIR.exists() and any(SCENE_DIR.rglob("*.glb"))
    print(f"\nHM3D scene files available: {has_scenes}")
else:
    print(f"Episode data not found at {content_dir}")
    print("Run: download ObjectNav episodes first (see habitat_objectnav_benchmark.ipynb)")

---
## 5. Single ObjectNav Step (with HM3D scenes)

This cell only runs if HM3D scene files are downloaded.

In [ ]:
from pathlib import Path

SCENE_DIR = Path("data/scene_datasets/hm3d")
has_scenes = SCENE_DIR.exists() and any(SCENE_DIR.rglob("*.glb"))

if has_scenes:
    from habitat.config import read_write
    
    config = habitat.get_config(
        config_path="benchmark/nav/objectnav/objectnav_hm3d.yaml"
    )
    with read_write(config):
        config.habitat.dataset.split = "val_mini"
        config.habitat.dataset.data_path = str(
            DATA_DIR / "{split}" / "{split}.json.gz"
        )
        scene_cfg = next(SCENE_DIR.rglob("*scene_dataset_config.json"), None)
        if scene_cfg:
            config.habitat.simulator.scene_dataset = str(scene_cfg)
        agent_cfg = config.habitat.simulator.agents.main_agent
        for sensor in ["rgb_sensor", "depth_sensor"]:
            s = getattr(agent_cfg.sim_sensors, sensor)
            s.width = 256
            s.height = 256

    env = habitat.Env(config=config)
    obs = env.reset()
    print(f"Episode: {env.current_episode.episode_id}")
    print(f"Scene:   {env.current_episode.scene_id}")
    print(f"Target:  {env.current_episode.object_category}")
    print(f"Obs keys: {list(obs.keys())}")
    for k, v in obs.items():
        if hasattr(v, 'shape'):
            print(f"  {k}: {v.shape} {v.dtype}")

    # Take one step forward
    obs = env.step(action=1)  # MOVE_FORWARD
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(obs["rgb"])
    axes[0].set_title(f"RGB — target: {env.current_episode.object_category}")
    axes[0].axis("off")
    axes[1].imshow(obs["depth"].squeeze(), cmap="plasma")
    axes[1].set_title("Depth")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    
    env.close()
    print("Environment closed.")
else:
    print("HM3D scenes not downloaded — skipping ObjectNav env test.")
    print("Download with:")
    print("  uv run python -m habitat_sim.utils.datasets_download \\")
    print("    --username <TOKEN_ID> --password <TOKEN_SECRET> \\")
    print("    --uids hm3d_minival_v0.2 --data-path data/")